# 07 — Noise exposure and CVD in Ohio counties: models

**Question:** Is the share of residents exposed to transportation noise associated with county CVD mortality (and CHD prevalence) in Ohio, after adjusting for confounders?

**Analysis plan (fixed before fitting any model):**

| Element | Primary analysis |
|---|---|
| Outcome | CVD deaths (ICD-10 I00–I99), 2018–2022 pooled, CDC WONDER |
| Age adjustment | Offset: log(expected deaths), by indirect standardization to Ohio 10-year age-specific rates |
| Exposure | % of residents at ≥50 dBA LAeq (building-weighted, BTS 2022/2023) |
| Covariates | % urban, % poverty, % non-Hispanic Black, smoking, PM2.5 |
| Model | Negative binomial regression (Poisson first, to test for overdispersion) |
| Effect measure | Rate ratio per 5 percentage points of exposure, with 95% CI |

**Secondary analyses:** exposure at ≥45 and ≥55 dBA (dose–response); CHD prevalence (PLACES) as outcome.

**Sensitivity analyses:** internal-point exposure (notebook 03); model without smoking; median income instead of poverty; % 65+ covariate with population offset instead of expected deaths.

**Diagnostics:** variance inflation factors; overdispersion; spatial autocorrelation of residuals.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")

def load_wonder(path):
    """Load a CDC WONDER export: keep data rows only (Notes empty), add numeric columns."""
    df = pd.read_csv(path, sep="\t", dtype=str)
    df = df[df["Notes"].isna()].drop(columns="Notes")
    for col in ["Deaths", "Population"]:
        if col in df.columns:
            df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")
    return df

In [2]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

def fit_nb(formula, offset_col):
    """Negative binomial: BFGS for a starting point, then Newton to converge precisely."""
    mod = smf.negativebinomial(formula, data=df, offset=df[offset_col])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)    # only the starting fit; Newton is checked below
        start = mod.fit(disp=False, maxiter=1000)
    return mod.fit(disp=False, method="newton", maxiter=200, start_params=start.params)

In [3]:
import os, requests
from dotenv import load_dotenv

load_dotenv("../.env")
CENSUS_API_KEY = os.environ["CENSUS_API_KEY"]

AGE = "Ten-Year Age Groups Code"
YEARS = 5   # 2018–2022

# ---- 1. Ohio age-specific CVD death rates (query D), with <1 and 1-4 merged into 0-4 ----
state = load_wonder(RAW / "D_ohio_age_2018_2022.txt")
state = state[state[AGE] != "NS"].copy()
state["age_group"] = state[AGE].replace({"1": "0-4", "1-4": "0-4"})
rates = state.groupby("age_group")[["Deaths_num", "Population_num"]].sum()
rates["rate"] = rates["Deaths_num"] / rates["Population_num"]          # deaths per person-year

# ---- 2. County populations by age from ACS table B01001 ----
# Line numbers for men; women are the same lines + 24 (e.g., men under 5 = line 3, women under 5 = line 27)
AGE_MAP = {
    "0-4":   [3],
    "5-14":  [4, 5],
    "15-24": [6, 7, 8, 9, 10],
    "25-34": [11, 12],
    "35-44": [13, 14],
    "45-54": [15, 16],
    "55-64": [17, 18, 19],
    "65-74": [20, 21, 22],
    "75-84": [23, 24],
    "85+":   [25],
}
def b01001(lines):
    return [f"B01001_{l:03d}E" for l in lines] + [f"B01001_{l + 24:03d}E" for l in lines]

assert set(rates.index) == set(AGE_MAP), (sorted(rates.index), list(AGE_MAP))

age_vars = [v for lines in AGE_MAP.values() for v in b01001(lines)]
r = requests.get(
    "https://api.census.gov/data/2022/acs/acs5",
    params={"get": ",".join(["NAME", "B01001_001E"] + age_vars),
            "for": "county:*", "in": "state:39", "key": CENSUS_API_KEY},
    timeout=60,
)
r.raise_for_status()
if "json" not in r.headers.get("Content-Type", ""):
    raise RuntimeError(f"Census API did not return JSON:\n{r.text[:500]}")
data = r.json()
acs_age_raw = pd.DataFrame(data[1:], columns=data[0])
acs_age_raw.to_csv(RAW / "acs5_2022_county_age_OH.csv", index=False)

num = acs_age_raw[["B01001_001E"] + age_vars].apply(pd.to_numeric)
acs_age = pd.DataFrame({"county_fips": acs_age_raw["state"] + acs_age_raw["county"]})
for group, lines in AGE_MAP.items():
    acs_age[group] = num[b01001(lines)].sum(axis=1)

assert len(acs_age) == 88
# Every person counted in exactly one age group: groups must add up to the county total
assert (acs_age[list(AGE_MAP)].sum(axis=1) == num["B01001_001E"]).all()

# ---- 3. Expected deaths = sum over age groups of (Ohio rate x county population x 5 years) ----
cty_pop = acs_age.melt(id_vars="county_fips", var_name="age_group", value_name="pop")
cty_pop = cty_pop.merge(rates["rate"], left_on="age_group", right_index=True,
                        how="left", validate="many_to_one")
cty_pop["expected"] = cty_pop["pop"] * cty_pop["rate"] * YEARS

expected = cty_pop.groupby("county_fips")["expected"].sum().reset_index()

ratio = expected["expected"].sum() / rates["Deaths_num"].sum()
print(f"Expected / observed (statewide): {ratio:.4f}")
assert abs(ratio - 1) < 0.03, ratio
expected["expected"] = expected["expected"] / ratio      # rescale so totals match exactly

print(f"Ohio CVD deaths 2018–2022 (age stated): {rates['Deaths_num'].sum():,.0f}")
rates

rates = rates.loc[list(AGE_MAP)]                       # put age groups in age order
rates["rate_per_100k"] = (rates["rate"] * 1e5).round(1)
rates

Expected / observed (statewide): 0.9993
Ohio CVD deaths 2018–2022 (age stated): 198,052


,Deaths_num,Population_num,rate,rate_per_100k
age_group,,,,
0-4,119,3405103.0,0.000035,3.5
5-14,46,7236207.0,0.000006,0.6
15-24,239,7563150.0,0.000032,3.2
25-34,1006,7765427.0,0.000130,13.0
35-44,3066,7119233.0,0.000431,43.1
45-54,8690,7179405.0,0.001210,121.0
55-64,23549,7939132.0,0.002966,296.6
65-74,37731,6132651.0,0.006152,615.2
75-84,48651,3031938.0,0.016046,1604.6


In [4]:
mort = load_wonder(RAW / "A_ohio_county_2018_2022.txt")
mort = mort.rename(columns={"County Code": "county_fips", "County": "county", "Deaths_num": "deaths",
                            "Population_num": "pop_wonder"})[["county_fips", "county", "deaths", "pop_wonder"]]

noise_b = pd.read_csv(PROCESSED / "noise_oh_county_2022_bldg.csv", dtype={"county_fips": str})
noise_p = pd.read_csv(PROCESSED / "noise_oh_county_2022.csv",      dtype={"county_fips": str})
pct = ["pct_ge45", "pct_ge50", "pct_ge55"]
noise_b = noise_b[["county_fips"] + pct].rename(columns={c: c.replace("pct_", "noise_") for c in pct})
noise_p = noise_p[["county_fips"] + pct].rename(columns={c: c.replace("pct_", "noisept_") for c in pct})

places = pd.read_csv(PROCESSED / "places_oh_county_ageadj.csv", dtype={"locationid": str}) \
           .rename(columns={"locationid": "county_fips"})
cov    = pd.read_csv(PROCESSED / "covariates_oh_county.csv", dtype={"county_fips": str})
pm     = pd.read_csv(PROCESSED / "pm25_oh_county.csv",       dtype={"county_fips": str})

df = mort
for other in [expected, noise_b, noise_p, places, cov, pm]:
    df = df.merge(other, on="county_fips", how="outer", validate="one_to_one")

assert len(df) == 88 and df.notna().all().all(), df.isna().sum()[df.isna().sum() > 0]

df["smr"] = df["deaths"] / df["expected"]
df.to_csv(PROCESSED / "analysis_oh_county.csv", index=False)
df[["deaths", "expected", "smr"]].describe().round(2)

,deaths,expected,smr
count,88.00,88.00,88.00
mean,2250.59,2250.59,1.04
std,3474.29,3523.32,0.15
min,216.00,195.37,0.62
25%,637.75,625.55,0.94
50%,1037.00,971.65,1.03
75%,2123.75,2122.88,1.15
max,23507.00,23148.90,1.38


In [5]:
predictors = ["noise_ge50", "pct_urban", "pct_poverty", "pct_black_nh", "pct_smoking", "pm25"]
X = sm.add_constant(df[predictors])
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
                index=predictors, name="VIF")
vif.round(2)

noise_ge50      1.11
pct_urban       2.82
pct_poverty     3.61
pct_black_nh    2.51
pct_smoking     4.50
pm25            1.95
Name: VIF, dtype: float64

In [6]:
df["log_expected"] = np.log(df["expected"])

# Exposure per 5 percentage points (the pre-specified effect scale)
for t in [45, 50, 55]:
    df[f"noise_ge{t}_5pp"] = df[f"noise_ge{t}"] * 100 / 5

# Covariate shares in percentage points (coefficients become "per 1 point")
for c in ["pct_urban", "pct_poverty", "pct_black_nh", "pct_smoking"]:
    df[f"{c}_pp"] = df[c] * 100

COVARS = "pct_urban_pp + pct_poverty_pp + pct_black_nh_pp + pct_smoking_pp + pm25"
formula = f"deaths ~ noise_ge50_5pp + {COVARS}"

In [7]:
pois = smf.glm(formula, data=df, family=sm.families.Poisson(),
               offset=df["log_expected"]).fit()

dispersion = pois.pearson_chi2 / pois.df_resid
print(f"Dispersion (Pearson chi-square / residual df): {dispersion:.1f}")

Dispersion (Pearson chi-square / residual df): 15.4


In [8]:
nb = smf.negativebinomial(formula, data=df, offset=df["log_expected"]).fit(disp=False)

def rr_table(res):
    """Rate ratios with 95% CIs, excluding the intercept and NB's dispersion parameter."""
    params = res.params.drop(["Intercept", "alpha"], errors="ignore")
    ci = res.conf_int().loc[params.index]
    return pd.DataFrame({
        "RR":      np.exp(params),
        "CI_low":  np.exp(ci[0]),
        "CI_high": np.exp(ci[1]),
        "p":       res.pvalues.loc[params.index],
    }).round(3)

print(f"NB dispersion parameter alpha = {nb.params['alpha']:.4f}")
rr_table(nb)

NB dispersion parameter alpha = 0.0093


,RR,CI_low,CI_high,p
noise_ge50_5pp,1.019,0.998,1.041,0.074
pct_urban_pp,1.001,1.000,1.002,0.137
pct_poverty_pp,1.004,0.994,1.014,0.464
pct_black_nh_pp,0.996,0.990,1.002,0.223
pct_smoking_pp,1.042,1.024,1.061,0.000
pm25,1.083,1.023,1.145,0.006


In [9]:
pd.DataFrame({"Poisson": rr_table(pois).loc["noise_ge50_5pp"],
              "Neg. binomial": rr_table(nb).loc["noise_ge50_5pp"]}).T

,RR,CI_low,CI_high,p
Poisson,1.009,1.004,1.014,0.001
Neg. binomial,1.019,0.998,1.041,0.074


## Primary result (pre-specified)

- **Overdispersion:** Poisson dispersion = 15.4, so negative binomial regression was used, as planned (alpha = 0.0093). Poisson would have understated uncertainty about 4-fold (Poisson 95% CI 1.004–1.014, p = 0.001).
- **Primary estimate:** each additional 5 percentage points of residents at ≥50 dBA was associated with a 1.9% higher age-adjusted CVD mortality rate (**RR 1.019, 95% CI 0.998–1.041, p = 0.074**), adjusted for % urban, % poverty, % non-Hispanic Black, smoking, and PM2.5.
- **Interpretation:** a positive but imprecise association; the confidence interval includes no association. Across the exposure IQR (~6.3 points), RR ≈ 1.024.
- **Note:** the NB estimate is larger than Poisson's (1.009), which weights the largest counties most heavily, suggesting the association may be stronger across smaller counties.

In [10]:
# Extra variables for the sensitivity analyses
df["noisept_ge50_5pp"] = df["noisept_ge50"] * 100 / 5
df["income_10k"]       = df["median_hh_income"] / 10_000
df["pct_65plus_pp"]    = df["pct_65plus"] * 100
df["log_pop"]          = np.log(df["pop_wonder"])     # person-years from query A

COVARS_NO_SMK = "pct_urban_pp + pct_poverty_pp + pct_black_nh_pp + pm25"
COVARS_INCOME = "pct_urban_pp + income_10k + pct_black_nh_pp + pct_smoking_pp + pm25"

specs = {
    # name: (formula, exposure term, offset)
    "Primary (>=50 dBA)":            (f"deaths ~ noise_ge50_5pp + {COVARS}", "noise_ge50_5pp", "log_expected"),
    "Unadjusted (>=50 dBA)":         ("deaths ~ noise_ge50_5pp",             "noise_ge50_5pp", "log_expected"),
    "Secondary: >=45 dBA":           (f"deaths ~ noise_ge45_5pp + {COVARS}", "noise_ge45_5pp", "log_expected"),
    "Secondary: >=55 dBA":           (f"deaths ~ noise_ge55_5pp + {COVARS}", "noise_ge55_5pp", "log_expected"),
    "Sens: internal-point exposure": (f"deaths ~ noisept_ge50_5pp + {COVARS}", "noisept_ge50_5pp", "log_expected"),
    "Sens: without smoking":         (f"deaths ~ noise_ge50_5pp + {COVARS_NO_SMK}", "noise_ge50_5pp", "log_expected"),
    "Sens: income instead of poverty": (f"deaths ~ noise_ge50_5pp + {COVARS_INCOME}", "noise_ge50_5pp", "log_expected"),
    "Sens: % 65+ with population offset": (f"deaths ~ noise_ge50_5pp + {COVARS} + pct_65plus_pp", "noise_ge50_5pp", "log_pop"),
}

rows = []
for name, (f, term, off) in specs.items():
    res = fit_nb(f, off)
    row = rr_table(res).loc[term].rename(name)
    row["converged"] = res.mle_retvals["converged"]
    rows.append(row)

results = pd.DataFrame(rows)
assert results["converged"].all(), results[~results["converged"]]
results.to_csv("../outputs/nb_results_noise.csv")
results

,RR,CI_low,CI_high,p,converged
Primary (>=50 dBA),1.019,0.998,1.041,0.074,True
Unadjusted (>=50 dBA),1.007,0.980,1.035,0.593,True
Secondary: >=45 dBA,1.007,0.998,1.017,0.129,True
Secondary: >=55 dBA,1.110,1.031,1.194,0.005,True
Sens: internal-point exposure,1.017,0.994,1.041,0.140,True
Sens: without smoking,1.021,0.997,1.045,0.084,True
Sens: income instead of poverty,1.010,0.989,1.032,0.343,True
Sens: % 65+ with population offset,1.023,1.002,1.045,0.035,True


In [11]:
IQR_PP = {t: (df[f"noise_ge{t}"].quantile(0.75) - df[f"noise_ge{t}"].quantile(0.25)) * 100
          for t in [45, 50, 55]}

for name, t in [("Primary (>=50 dBA)", 50), ("Secondary: >=45 dBA", 45), ("Secondary: >=55 dBA", 55)]:
    k = IQR_PP[t] / 5                     # number of 5-point steps in one IQR
    rr, lo, hi = results.loc[name, ["RR", "CI_low", "CI_high"]]
    print(f"{name}: per IQR ({IQR_PP[t]:.1f} pts) RR = {rr**k:.3f} ({lo**k:.3f}–{hi**k:.3f})")

Primary (>=50 dBA): per IQR (6.4 pts) RR = 1.024 (0.997–1.053)
Secondary: >=45 dBA: per IQR (16.6 pts) RR = 1.023 (0.993–1.057)
Secondary: >=55 dBA: per IQR (1.7 pts) RR = 1.036 (1.011–1.063)


In [12]:
import geopandas as gpd
from libpysal.weights import Queen
from esda.moran import Moran

# Pearson residuals from the primary NB model: (observed - predicted) / predicted SD
nb = fit_nb(formula, "log_expected")
mu = nb.predict()                                  # predicted deaths (uses the offset)
alpha = nb.params["alpha"]
df["resid_nb"] = (df["deaths"] - mu) / np.sqrt(mu + alpha * mu**2)

# County boundaries, matched to the analysis table
counties = gpd.read_file("../data/large/boundaries/cb_2022_us_county_500k.zip")
oh = counties.loc[counties["STATEFP"] == "39", ["GEOID", "geometry"]].rename(columns={"GEOID": "county_fips"})
gdf = oh.merge(df[["county_fips", "county", "smr", "resid_nb"]], on="county_fips",
               how="inner", validate="one_to_one")
assert len(gdf) == 88

# Neighbors: counties sharing a border or a corner ("queen" contiguity)
w = Queen.from_dataframe(gdf, use_index=False)
w.transform = "r"                                  # each county's neighbors weighted equally
assert not w.islands, w.islands

for col, label in [("smr", "Raw SMR"), ("resid_nb", "NB residuals (primary model)")]:
    mi = Moran(gdf[col], w, permutations=999)
    print(f"{label}: Moran's I = {mi.I:.3f}, permutation p = {mi.p_sim:.3f}")

Raw SMR: Moran's I = 0.236, permutation p = 0.002
NB residuals (primary model): Moran's I = 0.058, permutation p = 0.133


## Interpretation

### Data
- **Outcome:** 198,052 CVD deaths (I00–I99) in Ohio, 2018–2022. Expected deaths by indirect standardization to Ohio's age-specific rates (county age structure from ACS 2018–2022; expected/observed = 0.9993 before rescaling).
- County SMRs range from 0.62 to 1.38 (SD 0.15), far more variation than chance alone would produce.

### Primary result
- Poisson dispersion = 15.4, so negative binomial regression was used (alpha = 0.0093). A Poisson model would have understated uncertainty about 4-fold and reported p = 0.001.
- **Each additional 5 percentage points of residents at ≥50 dBA was associated with 1.9% higher age-adjusted CVD mortality (RR 1.019, 95% CI 0.998–1.041, p = 0.074).** Per interquartile range of exposure (6.4 points): RR 1.024 (0.997–1.053).
- A positive but imprecise association; the confidence interval includes no association.

### Secondary analyses: exposure thresholds (per IQR)
| Threshold | RR per IQR | 95% CI |
|---|---|---|
| ≥45 dBA | 1.023 | 0.993–1.057 |
| ≥50 dBA (primary) | 1.024 | 0.997–1.053 |
| ≥55 dBA | 1.036 | 1.011–1.063 |

Estimates are similar at 45 and 50 dBA and somewhat larger at 55 dBA, compatible with a modest dose–response, but the intervals overlap substantially.

### Sensitivity analyses (≥50 dBA, per 5 points)
- Every specification gave a positive estimate (RR 1.007–1.023), with overlapping intervals: the **direction is robust; the magnitude is imprecise.**
- Adjustment increased the estimate (unadjusted RR 1.007 → adjusted 1.019), suggesting the covariates were masking part of the association.
- Results were similar with internal-point exposure (1.017), without smoking (1.021), and with % 65+ and a population offset (1.023). Replacing poverty with median income reduced the estimate (1.010), the specification the result depends on most.
- All models converged (BFGS initialization, Newton refinement).

### Spatial autocorrelation
Raw SMRs are spatially clustered (Moran's I = 0.236, p = 0.002), but model residuals show little remaining clustering (I = 0.058, p = 0.133), so the covariates account for most of the geographic pattern. With 88 counties, weak residual clustering can't be ruled out.

### Limitations
- **Ecological design:** associations between counties don't show that noise-exposed individuals are those who die of CVD.
- **Limited power:** 88 counties allow only modest precision.
- **Exposure:** BTS excludes local streets and models urban roads at low speeds; noise year (2022/2023) is at the end of the mortality window.
- **Covariates:** PLACES smoking is model-based and correlated with SES; the result is sensitive to the choice of SES measure.

### Conclusion
Across Ohio counties, greater transportation noise exposure was associated with modestly higher age-adjusted CVD mortality, consistently in direction across specifications but with confidence intervals that generally include no association.